# Stock Relative Moves Report

Build market-report style tables directly from `stock_data.db` with:

- Day vs day change
- Week vs week change
- Month vs month change

Each ticker is evaluated on its own latest available trading date, so the notebook works even when some names are one session behind the rest of the market.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

DB_PATH = Path('../stock_data.db') if Path('../stock_data.db').exists() else Path('stock_data.db')
PRICE_COLUMN = 'adj_close'
WEEK_LOOKBACK_DAYS = 7


In [ ]:
def load_prices(db_path: Path) -> pd.DataFrame:
    query = '''
        SELECT
            ticker,
            date,
            COALESCE(adj_close, close) AS adj_close,
            close
        FROM ticker_prices
        ORDER BY ticker, date
    '''
    with sqlite3.connect(db_path) as conn:
        frame = pd.read_sql_query(query, conn)

    frame['date'] = pd.to_datetime(frame['date'])
    frame['adj_close'] = pd.to_numeric(frame['adj_close'], errors='coerce')
    frame['close'] = pd.to_numeric(frame['close'], errors='coerce')
    return frame.dropna(subset=['adj_close'])


def ticker_label(ticker: str) -> str:
    return ticker.replace('.CL', '')


def format_cop(value: float) -> str:
    if pd.isna(value):
        return '-'
    text = f"{value:,.2f}"
    return text.replace(',', 'X').replace('.', ',').replace('X', '.')


def get_anchor_price(series: pd.Series, anchor_date: pd.Timestamp):
    history = series.loc[series.index <= anchor_date].dropna()
    if history.empty:
        return pd.NA
    return history.iloc[-1]


def trailing_change(series: pd.Series, latest_date: pd.Timestamp, anchor_date: pd.Timestamp):
    latest_price = get_anchor_price(series, latest_date)
    anchor_price = get_anchor_price(series, anchor_date)
    if pd.isna(latest_price) or pd.isna(anchor_price) or anchor_price == 0:
        return pd.NA
    return latest_price / anchor_price - 1


def build_snapshot_table(price_table: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ticker in price_table.columns:
        series = price_table[ticker].dropna()
        if series.empty:
            continue

        latest_date = series.index.max()
        previous_date = series.index[series.index < latest_date].max() if (series.index < latest_date).any() else pd.NaT
        week_anchor = latest_date - pd.Timedelta(days=WEEK_LOOKBACK_DAYS)
        month_anchor = latest_date - pd.DateOffset(months=1)

        rows.append({
            'ticker': ticker_label(ticker),
            'as_of': latest_date.date().isoformat(),
            'last_price': series.loc[latest_date],
            'day_vs_day': trailing_change(series, latest_date, previous_date),
            'week_vs_week': trailing_change(series, latest_date, week_anchor),
            'month_vs_month': trailing_change(series, latest_date, month_anchor),
        })

    return pd.DataFrame(rows)


def style_report(frame: pd.DataFrame, pct_columns: list[str], sort_by: str | None = None):
    view = frame.copy()
    if sort_by is not None:
        view = view.sort_values(sort_by, ascending=False).reset_index(drop=True)

    formatters = {}
    if 'last_price' in view.columns:
        formatters['last_price'] = format_cop
    for col in pct_columns:
        formatters[col] = '{:+.2%}'

    styler = (
        view.style
        .format(formatters, na_rep='-')
        .set_properties(**{'text-align': 'right'})
        .set_properties(subset=['ticker'], **{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', '700'), ('font-size', '13px')]},
            {'selector': 'td', 'props': [('font-size', '12px')]},
        ])
        .hide(axis='index')
    )

    for col in pct_columns:
        if col not in view.columns:
            continue
        max_abs = view[col].abs().max(skipna=True)
        if pd.isna(max_abs) or max_abs == 0:
            continue
        styler = styler.bar(
            subset=[col],
            color=['#ff1f1f', '#68c17d'],
            align='mid',
            vmin=-max_abs,
            vmax=max_abs,
        )

    return styler


In [ ]:
prices_long = load_prices(DB_PATH)
price_table = prices_long.pivot_table(index='date', columns='ticker', values=PRICE_COLUMN, aggfunc='last').sort_index()
snapshot = build_snapshot_table(price_table)
latest_date = pd.to_datetime(snapshot['as_of']).max()

print(f'Latest trading date in DB: {latest_date.date()}')
print(f'Tickers included: {len(snapshot)}')

snapshot.sort_values('week_vs_week', ascending=False).head()

## Combined Market Snapshot

A single table with the latest price plus daily, weekly, and monthly percentage changes. The default ranking is `week_vs_week` to mimic a market-board view.

In [ ]:
display(style_report(snapshot[['ticker', 'as_of', 'last_price', 'day_vs_day', 'week_vs_week', 'month_vs_month']], pct_columns=['day_vs_day', 'week_vs_week', 'month_vs_month'], sort_by='week_vs_week'))

## Ranked Views By Horizon

Separate ranked tables for each horizon so the strongest and weakest names are easy to scan by time frame.

In [ ]:
for label, column in [
    ('Day vs Day', 'day_vs_day'),
    ('Week vs Week', 'week_vs_week'),
    ('Month vs Month', 'month_vs_month'),
]:
    print()
    print(label)
    report = snapshot[['ticker', 'as_of', 'last_price', column]].rename(columns={column: 'change'})
    display(style_report(report, pct_columns=['change'], sort_by='change'))